# P7 - Desbalance de clases (1:1, 1:5 y 1:10)

Experimento temporal para evaluar el efecto del desbalance manteniendo fijo el criterio principal de enlace (threshold=1). Las caracteristicas se calculan solo con historia; el horizonte futuro se usa exclusivamente para las etiquetas.

Genera muestras para razones negativos:positivos 1:1, 5:1 y 10:1 y guarda resumen y auditoria en `P7_desbalance_clases`.


In [1]:
from pathlib import Path
from itertools import combinations
from collections import Counter
import random
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.model_selection import train_test_split
import features as ft

BASE_DIR = Path('C:/Users/Dra. Pilar/UltimaversionProgramasComparacion')
OUT_DIR = BASE_DIR / 'P7_desbalance_clases'
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 41
random.seed(SEED)
np.random.seed(SEED)
MIN_COAUTHORED_PAPERS = 1
NEGATIVE_RATIOS = [1, 5, 10]
NEGATIVE_MODE = 'random'
VAL_SIZE = 0.20
MAX_POSITIVES_TRAINVAL = None
MAX_POSITIVES_TEST = None

FILES = {
    'full_2000_2020': BASE_DIR / 'BD_UNAM_Graph_2000_2020.gml',
    'sub_2000_2020':  BASE_DIR / 'BD_UNAM_Sub_Graph_2000_2020.gml',
    'full_2021_2022': BASE_DIR / 'BD_UNAM_Graph_2021_2022.gml',
    'sub_2021_2022':  BASE_DIR / 'BD_UNAM_Sub_Graph_2021_2022.gml',
    'full_2023_2024': BASE_DIR / 'BD_UNAM_Graph_2023_2024.gml',
    'sub_2023_2024':  BASE_DIR / 'BD_UNAM_Sub_Graph_2023_2024.gml',
}

missing = [str(p) for p in FILES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos GML:\n' + '\n'.join(missing))

G_2000_2020 = nx.read_gml(FILES['full_2000_2020'])
S_2000_2020 = nx.read_gml(FILES['sub_2000_2020'])
G_2021_2022 = nx.read_gml(FILES['full_2021_2022'])
S_2021_2022 = nx.read_gml(FILES['sub_2021_2022'])
G_2023_2024 = nx.read_gml(FILES['full_2023_2024'])
S_2023_2024 = nx.read_gml(FILES['sub_2023_2024'])

G_HIST_2022 = nx.compose(G_2000_2020, G_2021_2022)
S_HIST_2022 = nx.compose(S_2000_2020, S_2021_2022)

print('Grafos cargados:')
for name, G in [('G_2000_2020', G_2000_2020), ('G_2021_2022', G_2021_2022), ('G_2023_2024', G_2023_2024), ('G_HIST_2022', G_HIST_2022)]:
    print(f'{name}: nodos={G.number_of_nodes():,}, aristas={G.number_of_edges():,}')


def node_type(G, node):
    return str(G.nodes[node].get('tipo', '')).strip().lower()


def is_article_node(G, node):
    return node_type(G, node) == 'articulo'


def is_author_node(G, node):
    return node_type(G, node) == 'autor'


def author_nodes(G):
    # CRITICO: solo nodos marcados explicitamente como autores.
    # El grafo principal tambien contiene keywords y afiliaciones.
    return {n for n in G.nodes if is_author_node(G, n)}

def add_isolated_historical_authors(S_history, G_history):
    """Añade al subgrafo de coautoria los autores conocidos sin crear aristas."""
    S = S_history.copy()
    hist_authors = author_nodes(G_history)
    missing = hist_authors - set(S.nodes())
    S.add_nodes_from(missing)
    return S, missing


def canon_pair(u, v):
    return tuple(sorted((str(u), str(v))))


def coauthorship_counts(G, eligible_authors=None):
    eligible = None if eligible_authors is None else set(eligible_authors)
    counts = Counter()
    for article in G.nodes:
        if not is_article_node(G, article):
            continue
        authors = [
            a for a in G.neighbors(article)
            if is_author_node(G, a) and (eligible is None or a in eligible)
        ]
        authors = sorted(set(authors), key=str)
        for u, v in combinations(authors, 2):
            counts[canon_pair(u, v)] += 1
    return counts


def threshold_edges(G, threshold, eligible_authors=None):
    counts = coauthorship_counts(G, eligible_authors)
    return {pair for pair, n in counts.items() if n >= threshold}


def safe_feature(func, graph, u, v, default=np.nan):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else value
    except Exception:
        return default


def safe_len_feature(func, graph, u, v, default=0):
    try:
        if u not in graph or v not in graph:
            return default
        value = func(graph, u, v)
        return default if value is None else len(value)
    except Exception:
        return default


def paper_count(G, author):
    if author not in G:
        return 0
    return sum(1 for n in G.neighbors(author) if is_article_node(G, n))


def shortest_path_safe(G, u, v):
    try:
        if u not in G or v not in G:
            return np.nan
        return nx.shortest_path_length(G, u, v)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return np.nan


def historical_hardness(G_sub, u, v):
    if u not in G_sub or v not in G_sub:
        return False
    try:
        if len(list(nx.common_neighbors(G_sub, u, v))) > 0:
            return True
    except Exception:
        pass
    try:
        return nx.shortest_path_length(G_sub, u, v) <= 3
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return False


def build_forecast_pairs(G_history, G_future, S_history, threshold=1, negative_ratio=1, negative_mode='random', max_positives=None, seed=41):
    rng = random.Random(seed)
    hist_authors = author_nodes(G_history)
    future_authors = author_nodes(G_future)
    structural_authors = set(S_history.nodes())

    # Cohorte temporal: autores conocidos antes del horizonte y observados en el horizonte futuro.
    # NO se exige pertenecer al subgrafo histórico: no tener enlaces estructurales previos
    # es información legítima del estado histórico y no debe excluir al candidato.
    eligible = sorted(
        hist_authors.intersection(future_authors),
        key=str,
    )
    if len(eligible) < 2:
        raise ValueError('No hay suficientes autores elegibles para construir pares.')

    # Seguridad: ningun candidato puede ser keyword, afiliacion o articulo.
    bad_hist = [n for n in eligible if not is_author_node(G_history, n)]
    bad_future = [n for n in eligible if not is_author_node(G_future, n)]
    if bad_hist or bad_future:
        raise AssertionError('La cohorte contiene nodos que no son autores.')

    hist_positive = threshold_edges(G_history, threshold, eligible_authors=eligible)
    future_positive_all = threshold_edges(G_future, threshold, eligible_authors=eligible)
    future_positive = sorted(future_positive_all - hist_positive, key=lambda x: (x[0], x[1]))

    if max_positives is not None and len(future_positive) > max_positives:
        future_positive = rng.sample(future_positive, max_positives)

    positives = set(future_positive)
    n_neg = len(positives) * int(negative_ratio)
    if n_neg == 0:
        raise ValueError('No se encontraron positivos nuevos con el umbral elegido.')

    negatives = set()
    attempts = 0
    max_attempts = max(100000, n_neg * 500)

    while len(negatives) < n_neg and attempts < max_attempts:
        attempts += 1
        u, v = rng.sample(eligible, 2)
        pair = canon_pair(u, v)
        if pair in hist_positive or pair in positives or pair in negatives:
            continue
        if negative_mode == 'hard' and not historical_hardness(S_history, u, v):
            continue
        negatives.add(pair)

    if len(negatives) < n_neg:
        raise RuntimeError(f'No fue posible obtener {n_neg} negativos {negative_mode}; se obtuvieron {len(negatives)}.')

    rows = [(u, v, 1) for u, v in positives] + [(u, v, 0) for u, v in negatives]
    df = pd.DataFrame(rows, columns=['source', 'target', 'connected'])
    df = df.sample(frac=1, random_state=seed).reset_index(drop=True)

    audit = {
        'historical_authors': len(hist_authors),
        'future_authors': len(future_authors),
        'historical_structural_nodes': len(structural_authors),
        'historical_structural_active_authors': sum(1 for n in S_history.nodes if S_history.degree(n) > 0),
        'eligible_authors': len(eligible),
        'eligible_with_structural_links': sum(1 for n in eligible if n in S_history and S_history.degree(n) > 0),
        'eligible_structural_isolates': sum(1 for n in eligible if n in S_history and S_history.degree(n) == 0),
        'historical_positive_pairs': len(hist_positive),
        'future_positive_pairs_all': len(future_positive_all),
        'new_future_positive_pairs': len(positives),
        'negative_pairs': len(negatives),
        'negative_ratio': negative_ratio,
        'negative_mode': negative_mode,
        'threshold': threshold,
    }
    return df, audit


def add_historical_features(df, G_history, S_history):
    out = df.copy()

    # Todos los autores históricos deben existir en S_history; quienes no tenían
    # coautorías previas permanecen como nodos aislados.
    missing_structural = (set(out['source']) | set(out['target'])) - set(S_history.nodes())
    if missing_structural:
        raise AssertionError(f'Faltan {len(missing_structural)} autores históricos en S_history.')
    isolated_candidates = {n for n in (set(out['source']) | set(out['target'])) if S_history.degree(n) == 0}
    print(f'Autores candidatos estructuralmente aislados: {len(isolated_candidates):,}')

    # FP
    out['sum_of_papers'] = out.apply(lambda r: paper_count(G_history, r['source']) + paper_count(G_history, r['target']), axis=1)
    out['sum_of_keywords'] = out.apply(lambda r: safe_len_feature(ft.sum_keywords, G_history, r['source'], r['target'], 0), axis=1)
    out['keywords_match'] = out.apply(lambda r: safe_len_feature(ft.keyword_match, G_history, r['source'], r['target'], 0), axis=1)
    out['sum_of_areas'] = out.apply(lambda r: safe_len_feature(ft.sum_areas, G_history, r['source'], r['target'], 0), axis=1)

    # LSP
    out['sum_of_neighbors'] = out.apply(lambda r: len(ft.neighbors(G_history, r['source'])) + len(ft.neighbors(G_history, r['target'])) if r['source'] in G_history and r['target'] in G_history else 0, axis=1)
    out['log_secundary_neighbors'] = out.apply(lambda r: safe_feature(ft.sum_sec_neig, G_history, r['source'], r['target'], 0.0), axis=1)
    out['lenght_short_path'] = out.apply(lambda r: shortest_path_safe(S_history, r['source'], r['target']), axis=1)
    out['clustering_index_sum'] = out.apply(lambda r: safe_feature(ft.clustering_index, S_history, r['source'], r['target'], np.nan), axis=1)
    out['jaccard_coefficient'] = out.apply(lambda r: safe_feature(ft.jaccard_coefficient, S_history, r['source'], r['target'], 0.0), axis=1)
    out['resource_allocation'] = out.apply(lambda r: safe_feature(ft.get_resource_allocation_index, S_history, r['source'], r['target'], 0.0), axis=1)
    out['adamic_adar_index'] = out.apply(lambda r: safe_feature(ft.get_adamic_adar_index, S_history, r['source'], r['target'], 0.0), axis=1)
    out['preferential_attachment'] = out.apply(lambda r: safe_feature(ft.get_preferential_attachment, S_history, r['source'], r['target'], 0.0), axis=1)

    # GSP
    out['community_cn'] = out.apply(lambda r: safe_feature(ft.community_cn, S_history, r['source'], r['target'], 0.0), axis=1)
    out['community_ra'] = out.apply(lambda r: safe_feature(ft.community_ra, S_history, r['source'], r['target'], 0.0), axis=1)
    return out



# -----------------------------------------------------------------------------
# P7. Sensibilidad al desbalance de clases
# Mantiene fijo threshold=1 y cambia solo la relación negativos:positivos.
# Las características se calculan exclusivamente con el grafo histórico.
# -----------------------------------------------------------------------------

S_2000_2020_RAW = S_2000_2020.copy()
S_HIST_2022_RAW = S_HIST_2022.copy()
S_2000_2020, ISOLATED_2000_2020 = add_isolated_historical_authors(S_2000_2020_RAW, G_2000_2020)
S_HIST_2022, ISOLATED_2000_2022 = add_isolated_historical_authors(S_HIST_2022_RAW, G_HIST_2022)

print(f'Autores historicos anadidos como aislados 2000-2020: {len(ISOLATED_2000_2020):,}')
print(f'Autores historicos anadidos como aislados 2000-2022: {len(ISOLATED_2000_2022):,}')

FEATURE_COLUMNS = [
    'sum_of_papers', 'sum_of_neighbors', 'log_secundary_neighbors',
    'sum_of_keywords', 'keywords_match', 'sum_of_areas',
    'lenght_short_path', 'clustering_index_sum', 'jaccard_coefficient',
    'resource_allocation', 'adamic_adar_index', 'preferential_attachment',
    'community_cn', 'community_ra'
]

summary_rows = []
audit_rows = []

for ratio in NEGATIVE_RATIOS:
    print('\n' + '=' * 72)
    print(f'NEGATIVE RATIO = 1:{ratio}')
    print('=' * 72)

    # Desarrollo: historia 2000-2020 -> etiquetas 2021-2022
    development, audit_dev = build_forecast_pairs(
        G_2000_2020, G_2021_2022, S_2000_2020,
        threshold=MIN_COAUTHORED_PAPERS,
        negative_ratio=ratio,
        negative_mode=NEGATIVE_MODE,
        max_positives=MAX_POSITIVES_TRAINVAL,
        seed=SEED,
    )
    development = add_historical_features(development, G_2000_2020, S_2000_2020)

    class_counts = development['connected'].value_counts().sort_index()
    if len(class_counts) < 2 or class_counts.min() < 2:
        raise ValueError(f'1:{ratio}: no hay suficientes casos por clase: {class_counts.to_dict()}')

    sample_train, sample_val = train_test_split(
        development,
        test_size=VAL_SIZE,
        random_state=SEED,
        stratify=development['connected'],
    )
    sample_train = sample_train.reset_index(drop=True)
    sample_val = sample_val.reset_index(drop=True)

    # Test temporal: historia 2000-2022 -> etiquetas 2023-2024
    sample_test, audit_test = build_forecast_pairs(
        G_HIST_2022, G_2023_2024, S_HIST_2022,
        threshold=MIN_COAUTHORED_PAPERS,
        negative_ratio=ratio,
        negative_mode=NEGATIVE_MODE,
        max_positives=MAX_POSITIVES_TEST,
        seed=SEED + 1,
    )
    sample_test = add_historical_features(sample_test, G_HIST_2022, S_HIST_2022)

    required = {'source', 'target', 'connected', *FEATURE_COLUMNS}
    for split_name, df in [('train', sample_train), ('val', sample_val), ('test', sample_test)]:
        missing_cols = required - set(df.columns)
        if missing_cols:
            raise AssertionError(f'1:{ratio} {split_name}: faltan columnas {missing_cols}')
        if df.duplicated(['source', 'target']).any():
            raise AssertionError(f'1:{ratio} {split_name}: hay pares duplicados.')

    pairs_train = set(map(tuple, sample_train[['source', 'target']].to_numpy()))
    pairs_val = set(map(tuple, sample_val[['source', 'target']].to_numpy()))
    if not pairs_train.isdisjoint(pairs_val):
        raise AssertionError(f'1:{ratio}: hay pares repetidos entre train y val.')

    suffix = f'thr1_neg{ratio}_random_isolates'
    train_file = OUT_DIR / f'sample_temporal_train_hist2000_2020_target2021_2022_{suffix}.csv'
    val_file = OUT_DIR / f'sample_temporal_val_hist2000_2020_target2021_2022_{suffix}.csv'
    test_file = OUT_DIR / f'sample_temporal_test_hist2000_2022_target2023_2024_{suffix}.csv'

    sample_train.to_csv(train_file, index=False)
    sample_val.to_csv(val_file, index=False)
    sample_test.to_csv(test_file, index=False)

    for split_name, hist, horizon, audit in [
        ('train_val', '2000-2020', '2021-2022', audit_dev),
        ('test', '2000-2022', '2023-2024', audit_test),
    ]:
        audit_rows.append({
            'experiment': 'P7_class_imbalance',
            'ratio_negative_to_positive': f'{ratio}:1',
            'split': split_name,
            'feature_history': hist,
            'label_horizon': horizon,
            'threshold': MIN_COAUTHORED_PAPERS,
            'negative_mode': NEGATIVE_MODE,
            'seed': SEED if split_name == 'train_val' else SEED + 1,
            'future_used_for_features': False,
            **audit,
        })

    def counts(df):
        vc = df['connected'].value_counts()
        return int(vc.get(1, 0)), int(vc.get(0, 0))

    tr_p, tr_n = counts(sample_train)
    va_p, va_n = counts(sample_val)
    te_p, te_n = counts(sample_test)
    dev_p, dev_n = counts(development)

    summary_rows.append({
        'negative_ratio': ratio,
        'scenario': f'1:{ratio}',
        'threshold': MIN_COAUTHORED_PAPERS,
        'development_n': len(development),
        'development_positive': dev_p,
        'development_negative': dev_n,
        'development_positive_prevalence': dev_p / len(development),
        'train_n': len(sample_train),
        'train_positive': tr_p,
        'train_negative': tr_n,
        'val_n': len(sample_val),
        'val_positive': va_p,
        'val_negative': va_n,
        'test_n': len(sample_test),
        'test_positive': te_p,
        'test_negative': te_n,
        'test_positive_prevalence': te_p / len(sample_test),
        'train_file': train_file.name,
        'val_file': val_file.name,
        'test_file': test_file.name,
    })

    print(f'Desarrollo: n={len(development)}, positivos={dev_p}, negativos={dev_n}, prevalencia={dev_p/len(development):.4f}')
    print(f'Train:      n={len(sample_train)}, positivos={tr_p}, negativos={tr_n}')
    print(f'Val:        n={len(sample_val)}, positivos={va_p}, negativos={va_n}')
    print(f'Test:       n={len(sample_test)}, positivos={te_p}, negativos={te_n}, prevalencia={te_p/len(sample_test):.4f}')

summary_df = pd.DataFrame(summary_rows)
audit_df = pd.DataFrame(audit_rows)

summary_file = OUT_DIR / 'P7_class_imbalance_summary.csv'
audit_file = OUT_DIR / 'P7_class_imbalance_audit.csv'
summary_df.to_csv(summary_file, index=False)
audit_df.to_csv(audit_file, index=False)

print('\n' + '=' * 72)
print('RESUMEN DE DESBALANCE')
print('=' * 72)
print(summary_df[[
    'scenario', 'development_n', 'development_positive', 'development_negative',
    'test_n', 'test_positive', 'test_negative', 'test_positive_prevalence'
]].to_string(index=False))
print('\nArchivos resumen:')
print(summary_file)
print(audit_file)
print('\nCSV individuales guardados en:')
print(OUT_DIR)


C:\Users\pilarang\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Grafos cargados:
G_2000_2020: nodos=9,041, aristas=14,064
G_2021_2022: nodos=5,151, aristas=6,752
G_2023_2024: nodos=3,207, aristas=4,025
G_HIST_2022: nodos=13,371, aristas=20,728
Autores historicos anadidos como aislados 2000-2020: 0
Autores historicos anadidos como aislados 2000-2022: 0

NEGATIVE RATIO = 1:1
Autores candidatos estructuralmente aislados: 6
Autores candidatos estructuralmente aislados: 21
Desarrollo: n=34, positivos=17, negativos=17, prevalencia=0.5000
Train:      n=27, positivos=14, negativos=13
Val:        n=7, positivos=3, negativos=4
Test:       n=112, positivos=56, negativos=56, prevalencia=0.5000

NEGATIVE RATIO = 1:5
Autores candidatos estructuralmente aislados: 8
Autores candidatos estructuralmente aislados: 32
Desarrollo: n=102, positivos=17, negativos=85, prevalencia=0.1667
Train:      n=81, positivos=14, negativos=67
Val:        n=21, positivos=3, negativos=18
Test:       n=336, positivos=56, negativos=280, prevalencia=0.1667

NEGATIVE RATIO = 1:10
Autores c